# Machine Learning – From Scratch
### Based on Labs 2–7 & Week 8 Slides

| Lab | Topic |
|-----|-------|
| Lab 2 | Decision Tree (Entropy & Information Gain) |
| Lab 3 | Linear Regression & Logistic Regression |
| Lab 4 | K-Nearest Neighbors (Euclidean Distance) |
| Lab 5 | Naïve Bayes (Gaussian) |
| Lab 6 | Random Forest (Bagging + Feature Subsets) |
| Lab 7 / Week 8 | Evaluation Metrics (Confusion Matrix, Precision, Recall, F1) |

## Setup & Data Preparation

**Workflow (as taught in the slides):**
1. Load dataset
2. Define features `X` and target `y`
3. Train–Test Split
4. Feature Scaling (StandardScaler)
5. Build & evaluate each model

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, r2_score,
    confusion_matrix, precision_score, recall_score, f1_score
)

# ── Load Dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv("C:/Users/aliah/Downloads/archive/ENB2012_data.csv")

# Features: all columns except the last two targets
X = df.iloc[:, :-2].values

# ── Regression target (Y1 = Heating Load) ────────────────────────────────────
y_reg = df["Y1"].values

# ── Classification target: binary split around the median (Lab 3 approach) ───
# Round Y1 first, then binarise at the median so classes are balanced
y_cls = (np.round(y_reg) > np.median(np.round(y_reg))).astype(int)

# ── Train-Test Split (80/20, random_state=42) ─────────────────────────────────
X_train,   X_test,   y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_cls, test_size=0.2, random_state=42)

# ── Feature Scaling (needed for Linear/Logistic Regression & KNN) ─────────────
# Slides note: KNN is sensitive to scale -> always scale features
scaler = StandardScaler()
X_train   = scaler.fit_transform(X_train)
X_test    = scaler.transform(X_test)
X_train_c = scaler.fit_transform(X_train_c)
X_test_c  = scaler.transform(X_test_c)

print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")
print(f"Features         : {X_train.shape[1]}")
print(f"Class distribution (train): {np.bincount(y_train_c)}")

---
## Lab 3 – Linear Regression (from scratch)

**Goal:** Predict a continuous value (Heating Load Y1).  
**Method (Gradient Descent):**
$$\hat{y} = X \cdot w + b$$
$$w \leftarrow w - \alpha \cdot \frac{1}{n} X^T(\hat{y}-y), \quad b \leftarrow b - \alpha \cdot \overline{(\hat{y}-y)}$$

**Evaluation:** R² Score (how much variance the model explains)

In [ ]:
class LinearRegression:
    """
    Linear Regression via Gradient Descent.
    Lab 3 - predicts a continuous numeric target.
    """
    def __init__(self, lr=0.01, epochs=2000):
        self.lr     = lr
        self.epochs = epochs

    def fit(self, X, y):
        n, d       = X.shape
        self.w     = np.zeros(d)   # initialise weights to zero
        self.b     = 0.0           # initialise bias to zero
        self.loss_history = []

        for epoch in range(self.epochs):
            y_pred = X @ self.w + self.b
            error  = y_pred - y

            # Gradient of MSE w.r.t. w and b
            dw = (1/n) * (X.T @ error)
            db = np.mean(error)

            # Gradient descent update
            self.w -= self.lr * dw
            self.b -= self.lr * db

            # Track MSE every 200 epochs
            if epoch % 200 == 0:
                mse = np.mean(error ** 2)
                self.loss_history.append(mse)

    def predict(self, X):
        return X @ self.w + self.b


# ── Train & Evaluate ──────────────────────────────────────────────────────────
lr_model = LinearRegression(lr=0.01, epochs=2000)
lr_model.fit(X_train, y_train_r)
lr_pred  = lr_model.predict(X_test)
lr_r2    = r2_score(y_test_r, lr_pred)

print(f"Linear Regression  R² Score : {lr_r2:.4f}")

---
## Lab 3 – Logistic Regression (from scratch)

**Goal:** Classify into two categories (high / low heating load).  
**Method:** Use the **sigmoid** function to squash the linear output to [0, 1].  
$$\sigma(z) = \frac{1}{1+e^{-z}}$$
Binary Cross-Entropy loss gradient is identical in form to Linear Regression's MSE gradient — only the output changes.

In [ ]:
class LogisticRegression:
    """
    Logistic Regression via Gradient Descent.
    Lab 3 - binary classification using the sigmoid function.
    """
    def __init__(self, lr=0.001, epochs=5000):
        self.lr     = lr
        self.epochs = epochs

    def sigmoid(self, z):
        # Clip to avoid overflow in exp
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X, y):
        n, d   = X.shape
        self.w = np.zeros(d)
        self.b = 0.0

        for _ in range(self.epochs):
            z     = X @ self.w + self.b
            pred  = self.sigmoid(z)
            error = pred - y

            # Gradients (same form as linear regression)
            dw = (1/n) * (X.T @ error)
            db = np.mean(error)

            self.w -= self.lr * dw
            self.b -= self.lr * db

    def predict_proba(self, X):
        return self.sigmoid(X @ self.w + self.b)

    def predict(self, X):
        # Threshold at 0.5 -> binary prediction
        return (self.predict_proba(X) >= 0.5).astype(int)


# ── Train & Evaluate ──────────────────────────────────────────────────────────
log_model = LogisticRegression(lr=0.001, epochs=5000)
log_model.fit(X_train_c, y_train_c)
log_pred  = log_model.predict(X_test_c)

print(f"Logistic Regression  Accuracy : {accuracy_score(y_test_c, log_pred):.4f}")

---
## Lab 4 – K-Nearest Neighbors (from scratch)

**Steps (exactly as in Lab 4 slides):**
1. Choose **K**
2. Compute **Euclidean distance** between the new point and all training points  
   $d = \sqrt{\sum_i (x_i - x'_i)^2}$
3. Select **K closest** neighbours
4. **Majority vote** -> predicted class

> KNN is a *lazy learner* — it stores training data and computes at prediction time.  
> Feature scaling is **required** (done in setup).

In [ ]:
class KNN:
    """
    K-Nearest Neighbors - Lab 4.
    Uses Euclidean distance and majority-vote for classification.
    KNN does NOT learn a model — it stores training data (lazy learner).
    """
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        # Store training data - no learning happens here
        self.X_train = X
        self.y_train = y.astype(int)

    def _euclidean_distance(self, x1, x2):
        """Euclidean distance as shown in Lab 4 slides."""
        return np.sqrt(np.sum((x1 - x2) ** 2))

    def predict(self, X):
        predictions = []
        for x in X:
            # Step 2: compute distance to every training point
            distances = np.array([
                self._euclidean_distance(x, x_train)
                for x_train in self.X_train
            ])

            # Step 3: select K nearest neighbours
            k_indices = np.argsort(distances)[: self.k]
            k_labels  = self.y_train[k_indices]

            # Step 4: majority vote
            predictions.append(np.bincount(k_labels).argmax())

        return np.array(predictions)


# ── Train & Evaluate ──────────────────────────────────────────────────────────
knn_model = KNN(k=5)
knn_model.fit(X_train_c, y_train_c)
knn_pred  = knn_model.predict(X_test_c)

print(f"KNN (k=5)  Accuracy : {accuracy_score(y_test_c, knn_pred):.4f}")

---
## Lab 5 – Naïve Bayes – Gaussian (from scratch)

**Steps (Lab 5 slides):**
1. Calculate **class prior** $P(C)$
2. Calculate **feature likelihoods** using Gaussian PDF (for numeric data)
3. Multiply probabilities (in log-space to avoid underflow)
4. Choose the class with the **highest posterior**

$$P(x|C) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

> Slide note: Naïve Bayes assumes features are **independent** — even if not true, it works well in practice.

In [ ]:
class GaussianNaiveBayes:
    """
    Gaussian Naïve Bayes - Lab 5.
    Steps: (1) class prior  (2) feature likelihoods  (3) multiply  (4) argmax.
    Uses log-probabilities to avoid numerical underflow.
    """
    def fit(self, X, y):
        y = y.astype(int)
        self.classes = np.unique(y)
        self.mean    = {}   # mu per class
        self.var     = {}   # sigma^2 per class
        self.prior   = {}   # P(C)

        for c in self.classes:
            X_c = X[y == c]
            # Step 1 - class prior
            self.prior[c] = len(X_c) / len(X)
            # Step 2 - feature statistics
            self.mean[c]  = X_c.mean(axis=0)
            self.var[c]   = X_c.var(axis=0) + 1e-9   # epsilon for stability

    def _gaussian_log_likelihood(self, x, mean, var):
        """Log of Gaussian PDF for a single sample."""
        return -0.5 * np.sum(
            np.log(2 * np.pi * var) + ((x - mean) ** 2) / var
        )

    def predict(self, X):
        predictions = []
        for x in X:
            # Step 3 - compute log-posterior for each class
            log_posteriors = [
                np.log(self.prior[c]) +
                self._gaussian_log_likelihood(x, self.mean[c], self.var[c])
                for c in self.classes
            ]
            # Step 4 - choose class with highest posterior
            predictions.append(self.classes[np.argmax(log_posteriors)])
        return np.array(predictions)


# ── Train & Evaluate ──────────────────────────────────────────────────────────
nb_model = GaussianNaiveBayes()
nb_model.fit(X_train_c, y_train_c)
nb_pred  = nb_model.predict(X_test_c)

print(f"Gaussian Naïve Bayes  Accuracy : {accuracy_score(y_test_c, nb_pred):.4f}")

---
## Lab 2 – Decision Tree (from scratch)

**Steps (Lab 2 slides – ID3 style):**
1. Start with the entire dataset (Root Node)
2. Select the best feature using **Entropy & Information Gain**
3. Split the dataset on that feature
4. Repeat recursively for each subset
5. Stop when: all samples same class | max depth reached | no gain

$$\text{Entropy}(S) = -\sum_c p_c \log_2(p_c)$$
$$\text{Information Gain} = \text{Entropy}(S) - \sum_{v} \frac{|S_v|}{|S|} \text{Entropy}(S_v)$$

In [ ]:
class DecisionTree:
    """
    Decision Tree - Lab 2.
    Splitting criterion: Entropy & Information Gain (ID3 style).
    Stopping criteria: pure node | max depth | no valid split.
    """
    def __init__(self, max_depth=5):
        self.max_depth = max_depth

    # ── Entropy (Lab 2 formula) ──────────────────────────────────────────────
    def _entropy(self, y):
        _, counts = np.unique(y, return_counts=True)
        p = counts / len(y)
        # Avoid log(0) by filtering zero probabilities
        return -np.sum(p[p > 0] * np.log2(p[p > 0]))

    # ── Information Gain ────────────────────────────────────────────────────
    def _information_gain(self, y, left_y, right_y):
        n = len(y)
        weighted_entropy = (
            (len(left_y)  / n) * self._entropy(left_y) +
            (len(right_y) / n) * self._entropy(right_y)
        )
        return self._entropy(y) - weighted_entropy

    def _majority_class(self, y):
        return np.bincount(y.astype(int)).argmax()

    # ── Step 2: Find best feature and threshold ──────────────────────────────
    def _best_split(self, X, y):
        best_gain   = -1
        best_feat   = None
        best_thresh = None

        for feature in range(X.shape[1]):
            thresholds = np.unique(X[:, feature])
            for thresh in thresholds:
                left_mask  = X[:, feature] <= thresh
                right_mask = ~left_mask

                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue

                gain = self._information_gain(
                    y, y[left_mask], y[right_mask]
                )

                if gain > best_gain:
                    best_gain   = gain
                    best_feat   = feature
                    best_thresh = thresh

        return best_feat, best_thresh

    # ── Step 4: Build tree recursively ──────────────────────────────────────
    def _build(self, X, y, depth=0):
        # Step 5 - Stopping criteria
        if len(np.unique(y)) == 1:          # pure node
            return int(y[0])
        if depth >= self.max_depth:         # max depth reached
            return self._majority_class(y)

        feat, thresh = self._best_split(X, y)
        if feat is None:                    # no valid split
            return self._majority_class(y)

        left_mask  = X[:, feat] <= thresh
        right_mask = ~left_mask

        return {
            "feature"   : feat,
            "threshold" : thresh,
            "left"      : self._build(X[left_mask],  y[left_mask],  depth + 1),
            "right"     : self._build(X[right_mask], y[right_mask], depth + 1),
        }

    def fit(self, X, y):
        self.tree_ = self._build(X, y.astype(int))

    def _predict_one(self, x, node):
        if not isinstance(node, dict):
            return node
        if x[node["feature"]] <= node["threshold"]:
            return self._predict_one(x, node["left"])
        return self._predict_one(x, node["right"])

    def predict(self, X):
        return np.array([self._predict_one(x, self.tree_) for x in X])


# ── Train & Evaluate ──────────────────────────────────────────────────────────
dt_model = DecisionTree(max_depth=6)
dt_model.fit(X_train_c, y_train_c)
dt_pred  = dt_model.predict(X_test_c)

print(f"Decision Tree (max_depth=6)  Accuracy : {accuracy_score(y_test_c, dt_pred):.4f}")

---
## Lab 6 – Random Forest (from scratch)

**Lab 6 key points:**
- Random Forest = ensemble of many Decision Trees
- Each tree trains on a **random bootstrap sample** (Bagging)
- Each split considers only a **random subset of features** (reduces correlation between trees)
- Final prediction = **Majority vote** (classification) or Average (regression)

| | Decision Tree | Random Forest |
|---|---|---|
| Data | Whole dataset | Bootstrap sample |
| Features | All features | Random subset per split |
| Overfitting | High | Reduced |
| Speed | Fast | Slower |

In [ ]:
class RandomForestTree:
    """
    A single Decision Tree used inside Random Forest.
    Adds random feature subset selection at each split (key RF feature - Lab 6).
    Uses Entropy & Information Gain (consistent with Lab 2).
    """
    def __init__(self, max_depth=6, max_features=None):
        self.max_depth    = max_depth
        self.max_features = max_features  # number of features to consider per split

    def _entropy(self, y):
        _, counts = np.unique(y, return_counts=True)
        p = counts / len(y)
        return -np.sum(p[p > 0] * np.log2(p[p > 0]))

    def _information_gain(self, y, left_y, right_y):
        n = len(y)
        return self._entropy(y) - (
            (len(left_y)  / n) * self._entropy(left_y) +
            (len(right_y) / n) * self._entropy(right_y)
        )

    def _majority_class(self, y):
        return np.bincount(y.astype(int)).argmax()

    def _best_split(self, X, y):
        best_gain, best_feat, best_thresh = -1, None, None

        # ── Random feature subset (the RF trick from Lab 6) ──────────────────
        n_features      = X.shape[1]
        max_f           = self.max_features or int(np.sqrt(n_features))
        feature_indices = np.random.choice(n_features, max_f, replace=False)

        for feature in feature_indices:
            for thresh in np.unique(X[:, feature]):
                left_mask  = X[:, feature] <= thresh
                right_mask = ~left_mask
                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue
                gain = self._information_gain(y, y[left_mask], y[right_mask])
                if gain > best_gain:
                    best_gain, best_feat, best_thresh = gain, feature, thresh

        return best_feat, best_thresh

    def _build(self, X, y, depth=0):
        if len(np.unique(y)) == 1:
            return int(y[0])
        if depth >= self.max_depth:
            return self._majority_class(y)
        feat, thresh = self._best_split(X, y)
        if feat is None:
            return self._majority_class(y)
        left_mask  = X[:, feat] <= thresh
        right_mask = ~left_mask
        return {
            "feature"   : feat,
            "threshold" : thresh,
            "left"      : self._build(X[left_mask],  y[left_mask],  depth + 1),
            "right"     : self._build(X[right_mask], y[right_mask], depth + 1),
        }

    def fit(self, X, y):
        self.tree_ = self._build(X, y.astype(int))

    def _predict_one(self, x, node):
        if not isinstance(node, dict):
            return node
        if x[node["feature"]] <= node["threshold"]:
            return self._predict_one(x, node["left"])
        return self._predict_one(x, node["right"])

    def predict(self, X):
        return np.array([self._predict_one(x, self.tree_) for x in X])


class RandomForest:
    """
    Random Forest - Lab 6.
    Bagging: each tree trains on a bootstrap sample of the training data.
    Feature subsets: each split uses sqrt(n_features) random features.
    Final prediction: majority vote across all trees.
    """
    def __init__(self, n_trees=10, max_depth=6):
        self.n_trees   = n_trees
        self.max_depth = max_depth
        self.trees     = []

    def fit(self, X, y):
        y = y.astype(int)
        n = len(X)
        self.trees = []

        for _ in range(self.n_trees):
            # ── Bagging: bootstrap sample ─────────────────────────────────────
            indices = np.random.choice(n, n, replace=True)
            X_boot, y_boot = X[indices], y[indices]

            tree = RandomForestTree(max_depth=self.max_depth)
            tree.fit(X_boot, y_boot)
            self.trees.append(tree)

    def predict(self, X):
        # Collect predictions from all trees -> majority vote
        all_preds = np.array([tree.predict(X) for tree in self.trees])
        return np.array([
            np.bincount(all_preds[:, i].astype(int)).argmax()
            for i in range(X.shape[0])
        ])


# ── Train & Evaluate ──────────────────────────────────────────────────────────
rf_model = RandomForest(n_trees=10, max_depth=6)
rf_model.fit(X_train_c, y_train_c)
rf_pred  = rf_model.predict(X_test_c)

print(f"Random Forest (10 trees, depth=6)  Accuracy : {accuracy_score(y_test_c, rf_pred):.4f}")

---
## Lab 7 & Week 8 – Evaluation Metrics

Accuracy alone is **not enough** (Week 8 slides warning).  
We must report:

| Metric | Formula | When to use |
|--------|---------|-------------|
| **Accuracy** | (TP+TN)/Total | Balanced data |
| **Precision** | TP/(TP+FP) | Avoid false positives (e.g. spam filter) |
| **Recall** | TP/(TP+FN) | Avoid missing positives (e.g. disease detection) |
| **F1-Score** | 2·P·R/(P+R) | Balance both precision and recall |

$$\text{F1} = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

In [ ]:
def evaluate_classifier(name, y_true, y_pred):
    """
    Full evaluation as taught in Lab 7 & Week 8:
    Confusion Matrix -> Accuracy, Precision, Recall, F1-Score.
    """
    cm        = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    acc       = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall    = recall_score(y_true, y_pred, zero_division=0)
    f1        = f1_score(y_true, y_pred, zero_division=0)

    print(f"{'='*52}")
    print(f"  {name}")
    print(f"{'='*52}")
    print(f"  Confusion Matrix:")
    print(f"  {cm}")
    print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")
    print(f"  Accuracy  = {acc:.4f}   [ (TP+TN) / Total ]")
    print(f"  Precision = {precision:.4f}   [ TP / (TP+FP) ]")
    print(f"  Recall    = {recall:.4f}   [ TP / (TP+FN) ]")
    print(f"  F1-Score  = {f1:.4f}   [ 2*P*R / (P+R) ]")
    print()
    return {"Model": name, "Accuracy": acc, "Precision": precision,
            "Recall": recall, "F1": f1}


# ── Evaluate all classifiers ──────────────────────────────────────────────────
results = []
results.append(evaluate_classifier("Logistic Regression",  y_test_c, log_pred))
results.append(evaluate_classifier("KNN (k=5)",            y_test_c, knn_pred))
results.append(evaluate_classifier("Gaussian Naive Bayes", y_test_c, nb_pred))
results.append(evaluate_classifier("Decision Tree",        y_test_c, dt_pred))
results.append(evaluate_classifier("Random Forest",        y_test_c, rf_pred))

# ── Linear Regression result (R^2, not a classifier) ─────────────────────────
print(f"{'='*52}")
print(f"  Linear Regression  (Regression task - R^2 Score)")
print(f"{'='*52}")
print(f"  R^2 = {lr_r2:.4f}")
print()

---
## Full Model Comparison
Sorted by **F1-Score** (Lab 7 recommends F1 for general classification tasks).

In [ ]:
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="F1", ascending=False).reset_index(drop=True)

print(" FULL MODEL COMPARISON (sorted by F1-Score):\n")
print(df_results.to_string(index=False, float_format="{:.4f}".format))

best = df_results.iloc[0]
print(f"\n BEST OVERALL MODEL  ->  {best['Model']}")
print(f"   Accuracy  : {best['Accuracy']:.4f}")
print(f"   Precision : {best['Precision']:.4f}")
print(f"   Recall    : {best['Recall']:.4f}")
print(f"   F1-Score  : {best['F1']:.4f}")

---
## Built-in vs From-Scratch – Side-by-Side Comparison

This cell trains the **scikit-learn built-in** equivalents of every model built from scratch above,
then places both sets of results in a single comparison table.

| Model | From Scratch | Built-in (sklearn) |
|---|---|---|
| Linear Regression | Gradient Descent | `sklearn.linear_model.LinearRegression` |
| Logistic Regression | Sigmoid + GD | `sklearn.linear_model.LogisticRegression` |
| KNN | Euclidean + Vote | `sklearn.neighbors.KNeighborsClassifier` |
| Naïve Bayes | Gaussian PDF | `sklearn.naive_bayes.GaussianNB` |
| Decision Tree | ID3 Entropy | `sklearn.tree.DecisionTreeClassifier` |
| Random Forest | Bagging + Entropy | `sklearn.ensemble.RandomForestClassifier` |

> **Expected outcome:** sklearn models are generally equal or slightly better due to
> highly optimised solvers, but the from-scratch results should be close — confirming
> the correctness of the manual implementations.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Built-in vs From-Scratch Comparison
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.linear_model  import LinearRegression  as SklearnLR
from sklearn.linear_model  import LogisticRegression as SklearnLogReg
from sklearn.neighbors     import KNeighborsClassifier
from sklearn.naive_bayes   import GaussianNB
from sklearn.tree          import DecisionTreeClassifier
from sklearn.ensemble      import RandomForestClassifier
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── 1. Train all sklearn built-in models ─────────────────────────────────────

# Linear Regression (regression task -> R² score)
sk_lr = SklearnLR()
sk_lr.fit(X_train, y_train_r)
sk_lr_r2 = r2_score(y_test_r, sk_lr.predict(X_test))

# Logistic Regression
sk_log = SklearnLogReg(max_iter=5000, random_state=42)
sk_log.fit(X_train_c, y_train_c)
sk_log_pred = sk_log.predict(X_test_c)

# KNN
sk_knn = KNeighborsClassifier(n_neighbors=5)
sk_knn.fit(X_train_c, y_train_c)
sk_knn_pred = sk_knn.predict(X_test_c)

# Naive Bayes
sk_nb = GaussianNB()
sk_nb.fit(X_train_c, y_train_c)
sk_nb_pred = sk_nb.predict(X_test_c)

# Decision Tree
sk_dt = DecisionTreeClassifier(max_depth=6, criterion='entropy', random_state=42)
sk_dt.fit(X_train_c, y_train_c)
sk_dt_pred = sk_dt.predict(X_test_c)

# Random Forest
sk_rf = RandomForestClassifier(n_estimators=10, max_depth=6, random_state=42)
sk_rf.fit(X_train_c, y_train_c)
sk_rf_pred = sk_rf.predict(X_test_c)


# ── 2. Helper: collect metrics for a classifier ───────────────────────────────
def get_metrics(name, y_true, y_pred):
    return {
        "Model"     : name,
        "Accuracy"  : accuracy_score(y_true, y_pred),
        "Precision" : precision_score(y_true, y_pred, zero_division=0),
        "Recall"    : recall_score(y_true, y_pred, zero_division=0),
        "F1"        : f1_score(y_true, y_pred, zero_division=0),
    }


# ── 3. Build comparison DataFrame ────────────────────────────────────────────
scratch_rows = [
    get_metrics("Logistic Regression  [Scratch]",  y_test_c, log_pred),
    get_metrics("KNN (k=5)            [Scratch]",  y_test_c, knn_pred),
    get_metrics("Gaussian Naïve Bayes [Scratch]",  y_test_c, nb_pred),
    get_metrics("Decision Tree        [Scratch]",  y_test_c, dt_pred),
    get_metrics("Random Forest        [Scratch]",  y_test_c, rf_pred),
]

builtin_rows = [
    get_metrics("Logistic Regression  [sklearn]",  y_test_c, sk_log_pred),
    get_metrics("KNN (k=5)            [sklearn]",  y_test_c, sk_knn_pred),
    get_metrics("Gaussian Naïve Bayes [sklearn]",  y_test_c, sk_nb_pred),
    get_metrics("Decision Tree        [sklearn]",  y_test_c, sk_dt_pred),
    get_metrics("Random Forest        [sklearn]",  y_test_c, sk_rf_pred),
]

df_scratch = pd.DataFrame(scratch_rows)
df_builtin = pd.DataFrame(builtin_rows)

# ── 4. Print side-by-side table ───────────────────────────────────────────────
METRICS = ["Accuracy", "Precision", "Recall", "F1"]
MODEL_NAMES = ["Logistic Regression", "KNN (k=5)", "Gaussian Naïve Bayes",
               "Decision Tree", "Random Forest"]

print("=" * 88)
print(f"{'MODEL':<25} {'METRIC':<12} {'FROM SCRATCH':>14} {'SKLEARN':>14} {'DIFF':>10}")
print("=" * 88)

for i, name in enumerate(MODEL_NAMES):
    for j, metric in enumerate(METRICS):
        sc  = df_scratch.iloc[i][metric]
        sk  = df_builtin.iloc[i][metric]
        diff = sk - sc
        sign = "+" if diff >= 0 else ""
        label = name if j == 0 else ""
        print(f"  {label:<23} {metric:<12} {sc:>14.4f} {sk:>14.4f} {sign}{diff:>9.4f}")
    print("-" * 88)

# Linear Regression (R² only)
diff_r2 = sk_lr_r2 - lr_r2
sign_r2 = "+" if diff_r2 >= 0 else ""
print(f"  {'Linear Regression':<23} {'R² Score':<12} {lr_r2:>14.4f} {sk_lr_r2:>14.4f} {sign_r2}{diff_r2:>9.4f}")
print("=" * 88)


# ── 5. Bar chart: F1-Score comparison ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Built-in vs From-Scratch: Model Comparison", fontsize=14, fontweight="bold")

x      = np.arange(len(MODEL_NAMES))
width  = 0.35
colors = ("#4C72B0", "#DD8452")

for ax_idx, metric in enumerate(["F1", "Accuracy"]):
    ax = axes[ax_idx]
    sc_vals = [df_scratch.iloc[i][metric] for i in range(len(MODEL_NAMES))]
    sk_vals = [df_builtin.iloc[i][metric] for i in range(len(MODEL_NAMES))]

    bars1 = ax.bar(x - width/2, sc_vals, width, label="From Scratch", color=colors[0], alpha=0.85)
    bars2 = ax.bar(x + width/2, sk_vals, width, label="sklearn",       color=colors[1], alpha=0.85)

    ax.set_xlabel("Model", fontsize=11)
    ax.set_ylabel(metric, fontsize=11)
    ax.set_title(f"{metric} Score Comparison", fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels([n.split('(')[0].strip() for n in MODEL_NAMES], rotation=20, ha='right', fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.legend(fontsize=10)
    ax.yaxis.grid(True, linestyle='--', alpha=0.6)
    ax.set_axisbelow(True)

    # Annotate bars
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=7.5, color=colors[0])
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=7.5, color=colors[1])

plt.tight_layout()
plt.show()


# ── 6. Summary verdict ────────────────────────────────────────────────────────
print("\n COMPARISON SUMMARY")
print("=" * 60)
avg_diff = np.mean([
    abs(df_builtin.iloc[i]['F1'] - df_scratch.iloc[i]['F1'])
    for i in range(len(MODEL_NAMES))
])
print(f"  Average |F1 diff| across classifiers : {avg_diff:.4f}")
print(f"  Linear Regression R² diff (sklearn - scratch) : {diff_r2:+.4f}")
print()
for i, name in enumerate(MODEL_NAMES):
    sc_f1 = df_scratch.iloc[i]['F1']
    sk_f1 = df_builtin.iloc[i]['F1']
    winner = 'sklearn' if sk_f1 > sc_f1 else ('Scratch' if sc_f1 > sk_f1 else 'TIE')
    print(f"  {name:<22}  Scratch F1={sc_f1:.4f}  sklearn F1={sk_f1:.4f}  -> {winner}")
print("=" * 60)